Natural language processing: KeyBert

https://towardsdatascience.com/enhancing-keybert-keyword-extraction-results-with-keyphrasevectorizers-3796fa93f4db
https://github.com/MaartenGr/KeyBERT
https://pypi.org/project/bert-embedding/#:~:text=Bert%20Embeddings,from%20BERT's%20pre%2Dtrained%20model.
https://arxiv.org/pdf/1810.04805.pdf


PROBLEMAS:
- Ocupa demasiada RAM
- Deja de funcionar con 1000 propuestas laborales
- Lento

In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
# Dataset is now stored in a Pandas Dataframe
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/indeed_job_dataset_clean.csv')

Mounted at /content/drive


In [2]:
df = df[df['Description'].notna()]
words = " ".join(df['Description'].sample(n = 700))
#words = " ".join(df['Description'])
#print(words)

In [3]:
!pip install keyphrase-vectorizers
!pip install keybert
from keybert import KeyBERT
from keyphrase_vectorizers import KeyphraseCountVectorizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of spacy-curated-transformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.3/236.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.2/756.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 735.6/735.6 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.0/314.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

KeyBert con KeyphraseCountVectorizer como parámetro de vectorización.
KeyphraseCountVectorizer es un conjunto de vectorizadores que extraen frases clave con patrones de parte del discurso de una colección de documentos de texto y los convierten en una matriz de frases clave de documentos. Una matriz de frase clave de documento es una matriz matemática que describe la frecuencia de frases clave que aparecen en una colección de documentos. Las filas de la matriz indican los documentos de texto y las columnas indican las frases clave únicas.
https://github.com/TimSchopf/KeyphraseVectorizers

In [4]:
# Init KeyBERT
kw_model = KeyBERT()

phrases_set = set()
time_set = set()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# average word count for description:
# Split the text into individual descriptions
sampled_descriptions = df['Description'].sample(n=700)

# Count words in each description
word_counts = sampled_descriptions.str.split().str.len()

# Compute the average
average_words_per_description = word_counts.mean()

print("Average number of words per description:", average_words_per_description)

Average number of words per description: 241.3257142857143


In [6]:
import time
# Importing the library for RAM usage
import psutil

kw_model = KeyBERT()
l = pd.DataFrame()

for counter in range(10):
  start_time = time.time()

  words = ""
  words = " ".join(df['Description'].sample(n = 900))
  print(" ".join(words.split()[:10]))
  k = kw_model.extract_keywords(docs=words, vectorizer=KeyphraseCountVectorizer(), stop_words='english', use_mmr=True, diversity=0.7, top_n=20)

  print("--- %s seconds ---" % (time.time() - start_time))
  execution_time_seconds = round((time.time() - start_time), 2)
  time_set.add(execution_time_seconds)

  k.sort(key = lambda x: x[1], reverse = True)
  #k
  import pandas as pd
  import numpy as np

  np.random.seed(24)
  results = pd.DataFrame(k)
  # l.append(results)
  print(results)

  for i in results.iloc[:,0]:
    phrases_set.add(i)

  # Getting % usage of virtual_memory ( 3rd field)
  print('RAM memory % used:', psutil.virtual_memory()[2])
  # Getting usage of virtual_memory in GB ( 4th field)
  print('RAM Used (GB):', psutil.virtual_memory()[3]/1000000000)


extract prepare data raw data set senior data analyst extract
--- 217.61364102363586 seconds ---
                                                    0       1
0   complex large datasets programming skill scrip...  0.4687
1                amp implement improvement data model  0.2922
2                   current industrial research trend  0.2738
3   employment opportunity service men woman job type  0.2307
4                       report tableau reporting tool  0.2121
5                    transform load md sap deployment  0.2009
6                          global security leadership  0.1993
7                             limited linux operation  0.1953
8       pig scala spark knowledge jenkins chef puppet  0.1944
9   priority sale team influence direction ad prod...  0.1730
10                           creation method practice  0.1503
11  basic statistical concept probability distribu...  0.1274
12                                           wellness  0.1232
13                                 

KeyboardInterrupt: 

In [ ]:
len(phrases_set)

In [ ]:
phrases_set

In [ ]:
time_set

In [ ]:
import statistics
statistics.mean(time_set)

----------------PURE KEYBERT

In [ ]:
import sys
sys.exit()

In [ ]:
!pip install keybert
!pip install keybert[flair]
!pip install keybert[gensim]
!pip install keybert[spacy]
!pip install keybert[use]

KeyBert con n-gramas con rango de 3 palabras


In [ ]:
from keybert import KeyBERT
kw_model = KeyBERT()
#keywords = kw_model.extract_keywords(words)
#kw_model.extract_keywords(words, keyphrase_ngram_range=(1, 2), stop_words='english', use_maxsum=True, nr_candidates=10, top_n=10)

kw_model.extract_keywords(words, keyphrase_ngram_range=(1, 3))
